In [0]:
# Read the raw data from the Bronze table
df_orders_bronze_read = spark.table("workspace.my_database.orders_bronze")

# Clean column names by converting to lowercase and replacing spaces with underscores
df_orders_silver = df_orders_bronze_read
for col_name in df_orders_silver.columns:
    df_orders_silver = df_orders_silver.withColumnRenamed(col_name, col_name.replace(" ", "_").lower())

# Drop duplicate rows
df_orders_silver = df_orders_silver.dropDuplicates()

# Drop rows where 'order_id' OR 'total_amount' is null in a single step
df_orders_silver = df_orders_silver.dropna(subset=["order_id", "total_amount"])

# Replace null values for categorical columns
df_orders_silver_cleaned = df_orders_silver.fillna({
    "status": "Unknown",
    "category": "Unknown"
})

# Save the cleaned data into the Silver table
df_orders_silver_cleaned.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.my_database.orders_silver")

#  Updated the print message to reflect the actual logic
print("Orders Silver Layer cleaned and loaded successfully! (Null amounts dropped)")
display(df_orders_silver_cleaned)